
# Demo: Deploying and Testing an Agentic API with FastAPI

This notebook demonstrates how to deploy an AI agent inside Jupyter using **FastAPI**.
It simulates a real-world deployment scenario where an agent becomes accessible via REST API.


In [1]:
!pip uninstall torch torchvision torchaudio -y
!pip cache purge
!pip install torch --index-url https://download.pytorch.org/whl/cpu

Found existing installation: torch 2.13.0+cpu
Uninstalling torch-2.13.0+cpu:
  Successfully uninstalled torch-2.13.0+cpu


Files removed: 12 (123.6 MB)
Directories removed: 0
Looking in indexes: https://download.pytorch.org/whl/cpu
   ---------------------------------------- 0.0/121.9 MB ? eta -:--:--
   - -------------------------------------- 3.7/121.9 MB 21.8 MB/s eta 0:00:06
   -- ------------------------------------- 8.1/121.9 MB 21.0 MB/s eta 0:00:06
   ---- ----------------------------------- 14.7/121.9 MB 24.3 MB/s eta 0:00:05
   ------ --------------------------------- 20.7/121.9 MB 25.2 MB/s eta 0:00:05
   -------- ------------------------------- 26.5/121.9 MB 25.8 MB/s eta 0:00:04
   ---------- ----------------------------- 32.2/121.9 MB 26.2 MB/s eta 0:00:04
   ------------ --------------------------- 38.5/121.9 MB 26.9 MB/s eta 0:00:04
   -------------- ------------------------- 43.8/121.9 MB 26.5 MB/s eta 0:00:03
   --------------- ------------------------ 47.7/121.9 MB 25.5 MB/s eta 0:00:03
   ----------------- ---------------------- 54.0/121.9 MB 25.7 MB/s eta 0:00:03
   -------------------


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:

# Install dependencies (only needed once)
!pip install fastapi nest_asyncio uvicorn requests -q
!pip install langchain transformers sentencepiece -q

import nest_asyncio
import uvicorn
from fastapi import FastAPI, Request
import requests
from threading import Thread
import time

from langchain import PromptTemplate, LLMChain
from langchain.llms import HuggingFaceHub
from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
c:\study\AI\IITM_Agentic_AI_Training\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:

# Initialize Hugging Face model (local, free model)
model_name = "google/flan-t5-base"
hf_pipeline = pipeline("text2text-generation", model=model_name, max_length=128)
llm = HuggingFacePipeline(pipeline=hf_pipeline)

# Define prompt template
template = """
You are a helpful customer support agent.
Answer the user's question clearly and concisely.

Question: {query}
"""

prompt = PromptTemplate(input_variables=["query"], template=template)

# Build LLM Chain
support_agent = LLMChain(prompt=prompt, llm=llm)

# Define hybrid rule-based + LLM logic
def agent_response(query: str) -> str:
    q = query.lower()
    if "price" in q:
        return "Our pricing details are available on the website."
    elif "support" in q:
        return "You can reach support via email or chat."
    elif "refund" in q:
        return "Refunds are processed within 5–7 business days."
    else:
        response = support_agent.run(query)
        return response.strip()


Device set to use cpu
C:\Users\Ankur\AppData\Local\Temp\ipykernel_13728\3398284147.py:4: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFacePipeline`.
  llm = HuggingFacePipeline(pipeline=hf_pipeline)
C:\Users\Ankur\AppData\Local\Temp\ipykernel_13728\3398284147.py:17: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use RunnableSequence, e.g., `prompt | llm` instead.
  support_agent = LLMChain(prompt=prompt, llm=llm)


In [4]:

# Test agent logic before deployment
test_queries = [
    "Can you tell me about your pricing?",
    "How do I contact support?",
    "I want a refund for my order.",
    "How long does delivery take?",
    "What makes your product unique?"
]

for q in test_queries:
    print(f"User: {q}")
    print(f"Agent: {agent_response(q)}")
    print("-" * 60)


User: Can you tell me about your pricing?


C:\Users\Ankur\AppData\Local\Temp\ipykernel_13728\3398284147.py:29: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use invoke instead.
  response = support_agent.run(query)


Agent: You are a helpful customer support agent.
------------------------------------------------------------
User: How do I contact support?
Agent: You can reach support via email or chat.
------------------------------------------------------------
User: I want a refund for my order.
Agent: Refunds are processed within 5–7 business days.
------------------------------------------------------------
User: How long does delivery take?
Agent: a few minutes
------------------------------------------------------------
User: What makes your product unique?
Agent: It's made from recycled materials.
------------------------------------------------------------


In [5]:

# Create FastAPI app
app = FastAPI(title="Agent Deployment Demo")

@app.get("/")
def read_root():
    return {"message": "Agent API is running successfully!"}

@app.post("/predict")
async def predict(request: Request):
    data = await request.json()
    query = data.get("query", "")
    response = agent_response(query)
    return {"query": query, "response": response}


In [6]:

# Run FastAPI server inside Jupyter
nest_asyncio.apply()

def run_api():
    uvicorn.run(app, host="127.0.0.1", port=8000)

server_thread = Thread(target=run_api, daemon=True)
server_thread.start()

time.sleep(2)
print("✅ FastAPI server started on http://127.0.0.1:8000")


INFO:     Started server process [13728]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


✅ FastAPI server started on http://127.0.0.1:8000


In [7]:

# Test the API endpoint
url = "http://127.0.0.1:8000/predict"
payload = {"query": "How can I contact support?"}

response = requests.post(url, json=payload)
print("Response status:", response.status_code)
print("Response JSON:", response.json())


INFO:     127.0.0.1:59417 - "POST /predict HTTP/1.1" 200 OK
Response status: 200
Response JSON: {'query': 'How can I contact support?', 'response': 'You can reach support via email or chat.'}


In [8]:

# Test a few more queries
queries = [
    "Tell me about refund process.",
    "I want to know the price details.",
    "Who can I contact for support?",
    "Is there a discount available?"
]

for q in queries:
    res = requests.post(url, json={"query": q})
    print(f"User Query: {q}")
    print(f"Agent Response: {res.json()['response']}")
    print("-" * 50)


INFO:     127.0.0.1:53035 - "POST /predict HTTP/1.1" 200 OK
User Query: Tell me about refund process.
Agent Response: Refunds are processed within 5–7 business days.
--------------------------------------------------
INFO:     127.0.0.1:53036 - "POST /predict HTTP/1.1" 200 OK
User Query: I want to know the price details.
Agent Response: Our pricing details are available on the website.
--------------------------------------------------
INFO:     127.0.0.1:53037 - "POST /predict HTTP/1.1" 200 OK
User Query: Who can I contact for support?
Agent Response: You can reach support via email or chat.
--------------------------------------------------
INFO:     127.0.0.1:53038 - "POST /predict HTTP/1.1" 200 OK
User Query: Is there a discount available?
Agent Response: Yes.
--------------------------------------------------
